# 07. Descriptive statistics

The descriptive tables of Supplementary Note 1 (sample funnel, posts by country and language, taxonomy term frequencies, account concentration, monthly counts, filter precision). Converted from the script that produced them; reads the aggregated post files in `output/`.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))  # the modules the notebooks import live in src/


In [ ]:
"""Build the descriptive tables the referees asked for.

  python3 run_descriptives.py

Writes into output/tables/. Each file is named for the question it answers.
"""
import os, sys, time
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import numpy as np
import pandas as pd
import config as cfg
import aggregate as ag
import descriptives as ds

O = cfg.OUTPUT_DIR
T = os.path.join(O, "tables")
os.makedirs(T, exist_ok=True)
def save(df, name, **kw):
    df.to_csv(os.path.join(T, name), **kw)
    print(f"   -> tables/{name}")

t0 = time.time()
posts = pd.read_parquet(os.path.join(O, "climate_posts_2022.parquet"))
stance = pd.read_parquet(os.path.join(O, "stance_2022.parquet"))
posts = posts.merge(stance[["tweet_id", "stance_label", "stance_score"]], on="tweet_id")
posts = ag.add_party(posts)
strong = pd.read_parquet(os.path.join(O, "strong_flag.parquet")).strong.to_numpy()
posts["strong"] = strong
agg = pd.read_csv(os.path.join(O, "country_totals.csv"))
print(f"loaded {len(posts):,} posts in {time.time()-t0:.0f}s\n")


## 1. sample funnel (R2.1)


In [ ]:
print("1. sample funnel (R2.1)")
raw = pd.concat([pd.read_csv(f) for f in
                 __import__("glob").glob(os.path.join(O, "tweets_2022", "*.agg.csv"))],
                ignore_index=True)
funnel = pd.DataFrame([
    ("Posts in the Global-EC feed, 2022", int(raw.n.sum())),
    ("  geotagged", int(raw[raw.country_code.notna()].n.sum())),
    ("Match the Global Pulse taxonomy on the full text", int(raw.n_climate.sum())),
    ("  would match on the 140-character text", int(raw.n_climate_on_text.sum())),
    ("Geotagged, 2022, deduplicated (analysis sample)", len(posts)),
    ("  distinct authors", posts.user_id.nunique()),
    ("  in Parties with more than 100 posts", int(posts.party.isin(
        pd.read_csv(os.path.join(O, "indices_2022.csv"), index_col=0).index).sum())),
], columns=["step", "n"])
funnel["share_of_feed_pct"] = (100 * funnel.n / funnel.n.iloc[0]).round(4)
save(funnel, "T1_sample_funnel.csv", index=False)


## 2. per country (R2.1, R1.3.a, R2.2)


In [ ]:
# Referee 1 asks how many posts come from each country, so the unit here is the
# country, which is also the unit the figures plot and Table 1 regresses. The Party
# view is kept as a second table for anything that needs one row per NDC.
print("2. per country (R2.1, R1.3.a, R2.2)")


def by_unit(col):
    t = posts.groupby(col).agg(
        posts=("tweet_id", "size"),
        authors=("user_id", "nunique"),
        replies=("is_reply", "mean"),
        quotes=("is_quote", "mean"),
        retweets=("is_rt", "mean"),
        truncated=("was_truncated", "mean"),
        verified_posts=("verified", "mean"),
        with_coords=("has_coords", "mean"),
        believer=("stance_label", lambda s: s.str.startswith("believer").mean()),
        denier=("stance_label", lambda s: s.str.startswith("denier").mean()),
        strong_rule=("strong", "mean"),
    ).sort_values("posts", ascending=False)
    t["posts_per_author"] = (t.posts / t.authors).round(2)

    # The country's whole feed, not just its climate posts. This is what makes the
    # climate share below interpretable: a country can contribute many posts because
    # the platform is widely used there rather than because climate is discussed more.
    cc = posts.groupby(col).country_code.agg(
        lambda s: s.mode().iat[0] if len(s.mode()) else None)
    tot = agg.set_index("country_code")
    t["all_posts_in_country"] = cc.map(tot.n)
    t["all_authors_in_country"] = cc.map(tot.unique_authors_est)
    t["climate_share_of_country"] = (t.posts / t.all_posts_in_country).round(5)
    # The feed's language mix is deliberately not reported per country. It describes
    # what the vendor delivered, not what a country posts, and 99% English for Germany
    # would be read as a statement about German Twitter, which it is not. The
    # English-only nature of the corpus is stated once, in the Methods.
    for c in ["replies", "quotes", "retweets", "truncated", "verified_posts",
              "with_coords", "believer", "denier", "strong_rule"]:
        t[c] = (100 * t[c]).round(2)
    return t


by_country = by_unit("shp_country")
by_country["in_analysis"] = by_country.posts > cfg.MIN_TWEETS_PER_PARTY
by_country.index.name = "country"
save(by_country, "T2_by_country.csv")
save(by_unit("party"), "T2b_by_party.csv")

n_in = int(by_country.in_analysis.sum())
share = 100 * by_country.loc[by_country.in_analysis, "posts"].sum() / len(posts)
print(f"   {len(by_country)} countries in the corpus, {n_in} above the threshold "
      f"({share:.1f}% of posts)")
print(f"   median posts per included country: "
      f"{by_country.loc[by_country.in_analysis, 'posts'].median():,.0f}")


## 3. language mix (R2.2)


In [ ]:
print("3. language mix (R2.2)")
lang = pd.read_csv(os.path.join(O, "lang_by_country.csv"))
wide = lang.pivot_table(index="country_code", columns="lang", values="n",
                        aggfunc="sum", fill_value=0)
wide["total"] = wide.sum(axis=1)
for c in [c for c in wide.columns if c != "total"]:
    wide[f"{c}_pct"] = (100 * wide[c] / wide.total).round(3)
save(wide.sort_values("total", ascending=False), "T3_language_by_country.csv")
overall = lang.groupby("lang").n.sum().sort_values(ascending=False)
save((100 * overall / overall.sum()).round(3).to_frame("pct_of_all_posts"),
     "T3b_language_overall.csv")


## 4. taxonomy keyword frequencies (R2.1)


In [ ]:
print("4. taxonomy keyword frequencies (R2.1)")
phr = ds.taxonomy_phrases()
pos = phr[phr.role != "veto"].drop_duplicates("phrase")
counts = ds.count_phrases(posts.full_text.fillna("").tolist(), pos.phrase.tolist())
kw = pos.set_index("phrase").join(counts).reset_index()
kw["pct_of_sample"] = (100 * kw.posts / len(posts)).round(3)
save(kw.sort_values("posts", ascending=False), "T4_taxonomy_keywords.csv", index=False)


## 5. top accounts (R2.3, R1.3.c)


In [ ]:
print("5. top accounts (R2.3, R1.3.c)")
acc = posts.groupby(["user_id", "screen_name"]).agg(
    posts=("tweet_id", "size"),
    followers=("followers_count", "max"),
    friends=("friends_count", "max"),
    statuses=("statuses_count", "max"),
    verified=("verified", "max"),
    created=("user_created_at", "first"),
    description=("user_description", "first"),
    main_source=("source", lambda s: s.mode().iat[0] if len(s.mode()) else ""),
    main_country=("country", lambda s: s.mode().iat[0] if len(s.mode()) else ""),
).sort_values("posts", ascending=False).reset_index()
acc["share_of_sample_pct"] = (100 * acc.posts / len(posts)).round(4)
acc["posts_per_day_lifetime"] = ds.posting_rate(acc.statuses, acc.created).round(1)
acc["auto_guess"] = [ds.guess_account_type(d, s, src) for d, s, src
                     in zip(acc.description.fillna(""), acc.screen_name.fillna(""),
                            acc.main_source.fillna(""))]
acc["category"] = ""          # to be filled in by hand
top = acc.head(200)
save(top, "T5_top_200_accounts.csv", index=False)

conc = pd.DataFrame([
    ("distinct authors", acc.user_id.nunique(), np.nan),
    ("authors with a single post", int((acc.posts == 1).sum()),
     100 * (acc.posts == 1).mean()),
    ("posts by the top 10 accounts", int(acc.posts.head(10).sum()),
     100 * acc.posts.head(10).sum() / len(posts)),
    ("posts by the top 100 accounts", int(acc.posts.head(100).sum()),
     100 * acc.posts.head(100).sum() / len(posts)),
    ("posts by the top 1% of accounts", int(acc.posts.head(max(1, len(acc)//100)).sum()),
     100 * acc.posts.head(max(1, len(acc)//100)).sum() / len(posts)),
    ("posts from verified accounts", int(posts.verified.sum()),
     100 * posts.verified.mean()),
    ("posts from known automation tools",
     int(posts.source.isin(ds.AUTOMATION_SOURCES).sum()),
     100 * posts.source.isin(ds.AUTOMATION_SOURCES).mean()),
    ("retweets", int(posts.is_rt.sum()), 100 * posts.is_rt.mean()),
], columns=["measure", "n", "pct_of_posts"])
conc["pct_of_posts"] = conc.pct_of_posts.round(2)
save(conc, "T6_author_concentration.csv", index=False)

src = posts.source.value_counts().head(25).to_frame("posts")
src["pct"] = (100 * src.posts / len(posts)).round(2)
src["automation"] = src.index.isin(ds.AUTOMATION_SOURCES)
save(src, "T7_client_apps.csv")


## 6. monthly (R1.3.b)


In [ ]:
print("6. monthly volume and stance (R1.3.b)")
posts["month"] = pd.to_datetime(posts.created_at, utc=True, errors="coerce").dt.to_period("M")
monthly = posts.groupby("month").agg(
    posts=("tweet_id", "size"),
    authors=("user_id", "nunique"),
    mean_stance=("stance_score", "mean"),
    believer=("stance_label", lambda s: 100 * s.str.startswith("believer").mean()),
    denier=("stance_label", lambda s: 100 * s.str.startswith("denier").mean()),
).round(4)
save(monthly, "T8_monthly.csv")


## 7. filter precision (new)


In [ ]:
print("7. filter precision")
lab = pd.read_csv(os.path.join(O, "labelling_sample.csv"))
lab["is_climate"] = lab.is_climate.astype(str).str.strip()
import tweets_filter as tf
rows = []
for st_, name in [(1, "found by the 2025 run too"), (0, "only with the full text")]:
    s = lab[(lab._stratum == st_) & lab.is_climate.isin(["0", "1"])]
    k, n = int(s.is_climate.astype(int).sum()), len(s)
    lo, hi = tf.wilson_ci(k, n)
    rows.append((name, n, k, round(k/n, 4), round(lo, 4), round(hi, 4)))
prec = pd.DataFrame(rows, columns=["stratum", "labelled", "climate", "precision",
                                   "ci_low", "ci_high"])
save(prec, "T9_filter_precision.csv", index=False)

print(f"\n{len(os.listdir(T))} tables in {T}")
print(f"{(time.time()-t0)/60:.1f} min")
